In [5]:
import os

# Point to the PARENT folder that CONTAINS both IEHHR_test and IEHHR_train
dataset_root = r'D:\GSoc 2026\Human AI - RenAIssance Test1\Data\IEHHR Dataset'

# Quick sanity check
for split in os.listdir(dataset_root):
    split_path = os.path.join(dataset_root, split)
    if os.path.isdir(split_path):
        records = [d for d in os.listdir(split_path) 
                   if os.path.isdir(os.path.join(split_path, d))]
        print(f"{split}: {len(records)} top-level entries")

.git: 5 top-level entries
IEHHR_test: 1 top-level entries
IEHHR_training_part1: 374 top-level entries
IEHHR_training_part2: 291 top-level entries
IEHHR_training_part3: 303 top-level entries
kraken_env: 4 top-level entries
kraken_env_311: 4 top-level entries


In [6]:
import os

splits = {
    'test': 'IEHHR_test',
    'train_p1': 'IEHHR_training_part1',
    'train_p2': 'IEHHR_training_part2',
    'train_p3': 'IEHHR_training_part3',
}

for split_name, split_folder in splits.items():
    split_path = os.path.join(dataset_root, split_folder)
    
    line_images    = []
    word_images    = []
    transcriptions = []
    categories     = set()
    
    for root, dirs, files in os.walk(split_path):
        for f in files:
            if f.endswith('.png') and 'Line' in f and 'Word' not in f:
                line_images.append(f)
            elif f.endswith('.png') and 'Word' in f:
                word_images.append(f)
            elif f.endswith('_transcription.txt'):
                transcriptions.append(os.path.join(root, f))
            elif f.endswith('_category.txt'):
                with open(os.path.join(root, f), 'r', encoding='utf-8') as c:
                    categories.add(c.read().strip())

    print(f"\n{split_name}")
    print(f"  Line images    : {len(line_images)}")
    print(f"  Word images    : {len(word_images)}")
    print(f"  Transcriptions : {len(transcriptions)}")
    print(f"  Categories     : {categories}")


test
  Line images    : 757
  Word images    : 8026
  Transcriptions : 0
  Categories     : set()

train_p1
  Line images    : 1199
  Word images    : 12162
  Transcriptions : 748
  Categories     : {'idPage10391_Record10_Line0_Word0:other\nidPage10391_Record10_Line0_Word1:other\nidPage10391_Record10_Line0_Word2:other\nidPage10391_Record10_Line0_Word3:other\nidPage10391_Record10_Line0_Word4:other\nidPage10391_Record10_Line0_Word5:name\nidPage10391_Record10_Line0_Word6:surname\nidPage10391_Record10_Line0_Word7:occupation\nidPage10391_Record10_Line0_Word8:other\nidPage10391_Record10_Line0_Word9:location\nidPage10391_Record10_Line0_Word10:location\nidPage10391_Record10_Line0_Word11:location\nidPage10391_Record10_Line1_Word0:other\nidPage10391_Record10_Line1_Word1:other\nidPage10391_Record10_Line1_Word2:location\nidPage10391_Record10_Line1_Word3:other\nidPage10391_Record10_Line1_Word4:name\nidPage10391_Record10_Line1_Word5:state\nidPage10391_Record10_Line1_Word6:other\nidPage10391_Record1

In [7]:
# Just count unique categories, don't print all of them
all_categories = set()
for split in splits:
    split_path = os.path.join(dataset_root, split)
    for root, dirs, files in os.walk(split_path):
        for f in files:
            if f.endswith('_category.txt'):
                val = open(os.path.join(root,f), 
                          encoding='utf-8').read().strip()
                all_categories.add(val)

print(f"Number of unique categories: {len(all_categories)}")
print("Categories:")
for c in sorted(all_categories):
    print(f"  '{c}'")

Number of unique categories: 0
Categories:


In [8]:
import random

all_transcriptions = []
for split_folder in splits.values():
    split_path = os.path.join(dataset_root, split_folder)
    for root, dirs, files in os.walk(split_path):
        for f in files:
            if f.endswith('_transcription.txt'):
                all_transcriptions.append(os.path.join(root, f))

print(f"Total transcription files: {len(all_transcriptions)}\n")

for path in random.sample(all_transcriptions, 5):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read().strip()
    # Also read its category
    cat_path = path.replace('_transcription.txt', '_category.txt')
    category = open(cat_path).read().strip() if os.path.exists(cat_path) else 'unknown'
    
    print(f"Record   : {os.path.basename(os.path.dirname(path))}")
    print(f"Category : {category}")
    print(f"Text     : {text}")
    print("---")

Total transcription files: 1936

Record   : lines
Category : idPage10465_Record4_Line0:other other other other other name surname occupation other location other other
idPage10465_Record4_Line1:location location location location other other name surname occupation other other name
idPage10465_Record4_Line2:other other name state other other name surname occupation other
idPage10465_Record4_Line3:location location location location other other other name
Text     : idPage10465_Record4_Line0:Dilluns a 3 rebere de Jaume Xifre pages de Gualba habitant en
idPage10465_Record4_Line1:St Esteva de Vilanova fill de Antoni Xifre pages y de Magdalena
idPage10465_Record4_Line2:defuncts ab Maryanna donsella filla de Pere Guilla pages de
idPage10465_Record4_Line3:St Esteva de Vilanova defunct y de Antiga
---
Record   : lines
Category : idPage10372_Record10_Line0:other other other other name surname occupation occupation occupation other other
idPage10372_Record10_Line1:location other other name surn

In [9]:
import os
import json
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class WordSample:
    image_path: str
    category:   str        # name/surname/occupation/location/state/other
    line_idx:   int
    word_idx:   int

@dataclass
class LineSample:
    image_path:  str
    line_idx:    int
    words:       list[WordSample] = field(default_factory=list)

@dataclass
class Record:
    record_id:      str
    page_id:        str
    lines:          list[LineSample] = field(default_factory=list)
    transcription:  Optional[str] = None   # None for test set
    categories:     Optional[str] = None

In [ ]:
def parse_category_file(cat_text: str) -> dict:
    """
    Parse category file into a dict mapping
    'LineN_WordM' → category_label.

    Category file can be either:
    idPage_RecordX_Line0_Word0:other
    idPage_RecordX_Line0_Word1:name
    ...
    OR line-level:
    idPage_RecordX_Line0:other other name surname

    For line-level entries, this function expands the label list to per-word keys
    so downstream code can always look up labels as LineN_WordM.
    """
    mapping = {}
    for line in cat_text.strip().split('\n'):
        line = line.strip()
        if not line or ':' not in line:
            continue
        key, val = line.split(':', 1)
        val = val.strip()

        # Extract just LineN or LineN_WordM from the full ID
        parts = key.split('_')
        short_key = '_'.join(p for p in parts if p.startswith('Line') or p.startswith('Word'))

        # If this is a line-level category string, expand into per-word labels
        if short_key.startswith('Line') and '_Word' not in short_key:
            labels = val.split()
            for word_idx, label in enumerate(labels):
                word_key = f"{short_key}_Word{word_idx}"
                mapping[word_key] = label
            # keep line-level mapping as a fallback too
            mapping[short_key] = val
        else:
            mapping[short_key] = val
    return mapping

In [ ]:
def load_split(split_path: str) -> list[Record]:
    """Load all records from one dataset split folder."""
    records = []
    
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return records
    
    for record_name in sorted(os.listdir(split_path)):
        record_path = os.path.join(split_path, record_name)
        if not os.path.isdir(record_path):
            continue
        
        # Parse page ID from record name
        # Format: idPageXXXX_RecordN
        parts = record_name.split('_')
        page_id   = parts[0] if parts else record_name
        
        record = Record(record_id=record_name, page_id=page_id)
        
        # ── Read transcription ──────────────────────────────────────
        lines_path = os.path.join(record_path, 'lines')
        if os.path.exists(lines_path):
            for f in os.listdir(lines_path):
                if f.endswith('_transcription.txt'):
                    full = os.path.join(lines_path, f)
                    record.transcription = open(
                        full, encoding='utf-8').read().strip()
                elif f.endswith('_category.txt'):
                    full = os.path.join(lines_path, f)
                    record.categories = open(
                        full, encoding='utf-8').read().strip()
        
        # ── Load line images ────────────────────────────────────────
        if os.path.exists(lines_path):
            line_imgs = sorted([
                f for f in os.listdir(lines_path)
                if f.endswith('.png') and 'Line' in f
            ])
            for img_file in line_imgs:
                # Extract line index from filename
                # Format: idPageXXXX_RecordN_LineM.png
                line_idx = int(img_file.split('Line')[1].replace('.png',''))
                line_sample = LineSample(
                    image_path=os.path.join(lines_path, img_file),
                    line_idx=line_idx
                )
                record.lines.append(line_sample)
        
        # ── Load word images ────────────────────────────────────────
        words_path = os.path.join(record_path, 'words')
        if os.path.exists(words_path):
            # Read word-level category file if present
            word_cat_map = {}
            for f in os.listdir(words_path):
                if f.endswith('_category.txt'):
                    full = os.path.join(words_path, f)
                    cat_text = open(full, encoding='utf-8').read()
                    word_cat_map = parse_category_file(cat_text)
            
            word_imgs = sorted([
                f for f in os.listdir(words_path)
                if f.endswith('.png') and 'Word' in f
            ])
            for img_file in word_imgs:
                # Format: idPageXXXX_RecordN_LineM_WordK.png
                after_line = img_file.split('Line')[1].replace('.png','')
                line_part, word_part = after_line.split('_Word')
                line_idx = int(line_part)
                word_idx = int(word_part)

                short_key = f'Line{line_idx}_Word{word_idx}'
                category = word_cat_map.get(short_key)
                if category is None:
                    # fallback to line-level classification if per-word is not present
                    line_key = f'Line{line_idx}'
                    line_val = word_cat_map.get(line_key, '').split()
                    if 0 <= word_idx < len(line_val):
                        category = line_val[word_idx]
                    else:
                        category = 'other'

                word_sample = WordSample(
                    image_path=os.path.join(words_path, img_file),
                    category=category,
                    line_idx=line_idx,
                    word_idx=word_idx
                )
                # Attach word to its line
                for line in record.lines:
                    if line.line_idx == line_idx:
                        line.words.append(word_sample)
                        break
        
        
        records.append(record)
    
    return records

In [12]:
# ── Load all splits ──────────────────────────────────────────────────────────
dataset_root = r'D:\GSoc 2026\Human AI - RenAIssance Test1\Data\IEHHR Dataset'

print("Loading splits...")
test_records   = load_split(os.path.join(dataset_root, 'IEHHR_test'))
train_records  = (
    load_split(os.path.join(dataset_root, 'IEHHR_training_part1')) +
    load_split(os.path.join(dataset_root, 'IEHHR_training_part2')) +
    load_split(os.path.join(dataset_root, 'IEHHR_training_part3'))
)

print(f"Train records : {len(train_records)}")
print(f"Test records  : {len(test_records)}")

# Sanity check one record
r = train_records[0]
print(f"\nSample record  : {r.record_id}")
print(f"Lines          : {len(r.lines)}")
print(f"Words in line 0: {len(r.lines[0].words) if r.lines else 0}")
print(f"Transcription  : {r.transcription[:80] if r.transcription else None}")
print(f"Has categories : {r.categories is not None}")

Loading splits...
Train records : 968
Test records  : 1

Sample record  : idPage10354_Record1
Lines          : 5
Words in line 0: 12
Transcription  : idPage10354_Record1_Line0:Pro al pr de dits mes y any rebere de las sposallas
id
Has categories : True


In [13]:
test_path = os.path.join(dataset_root, 'IEHHR_test')
print(os.listdir(test_path))

['Readme_EVALUATION.pdf', 'Records']


In [14]:
# Check if there's a Records subfolder
records_subfolder = os.path.join(test_path, 'Records')
if os.path.exists(records_subfolder):
    test_records = load_split(records_subfolder)
else:
    test_records = load_split(test_path)

print(f"Test records: {len(test_records)}")

Test records: 253


In [15]:
def parse_transcription(trans_text: str) -> dict[str, str]:
    """
    Convert transcription file content into a dict:
    'Line0' → 'Pro al pr de dits mes y any...'
    'Line1' → 'next line text...'
    """
    result = {}
    if not trans_text:
        return result
    
    for entry in trans_text.strip().split('\n'):
        entry = entry.strip()
        if not entry or ':' not in entry:
            continue
        
        key, text = entry.split(':', 1)
        # Extract just 'Line0', 'Line1' etc from full ID
        # key looks like: idPage10354_Record1_Line0
        if 'Line' in key:
            line_part = 'Line' + key.split('Line')[1]
            result[line_part] = text.strip()
    
    return result


# Test it on your sample record
r = train_records[0]
parsed = parse_transcription(r.transcription)
print("Parsed transcription:")
for line_key, text in parsed.items():
    print(f"  {line_key}: {text}")

Parsed transcription:
  Line0: Pro al pr de dits mes y any rebere de las sposallas
  Line1: de Antoni Colomer assaunador de Bara fill de Genis
  Line2: Colomer pages de Vilassar y de Antiga ab Hieronyma
  Line3: donsella filla de Onofre Esquer morraler de Bara y
  Line4: de Eularia


In [16]:
def compute_cer(hypothesis: str, reference: str) -> float:
    """
    Character Error Rate = edit_distance(hyp, ref) / len(ref)
    
    Edit distance counts the minimum number of character-level
    insertions, deletions, substitutions needed to turn
    hypothesis into reference.
    """
    # Clean both strings
    hyp = hypothesis.strip().lower()
    ref = reference.strip().lower()
    
    if len(ref) == 0:
        return 0.0
    
    # Build the DP table for edit distance
    # dp[i][j] = edit distance between hyp[:i] and ref[:j]
    dp = [[0] * (len(ref) + 1) for _ in range(len(hyp) + 1)]
    
    # Base cases
    for i in range(len(hyp) + 1):
        dp[i][0] = i      # delete all hyp chars
    for j in range(len(ref) + 1):
        dp[0][j] = j      # insert all ref chars
    
    # Fill table
    for i in range(1, len(hyp) + 1):
        for j in range(1, len(ref) + 1):
            if hyp[i-1] == ref[j-1]:
                dp[i][j] = dp[i-1][j-1]          # match, no cost
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )
    
    return dp[len(hyp)][len(ref)] / len(ref)

In [18]:
import pytesseract
from PIL import Image

# Set path to tesseract executable — adjust if yours differs
pytesseract.pytesseract.tesseract_cmd = (
    r'C:\Program Files\Tesseract-OCR\tesseract.exe'
)

def ocr_line_tesseract(image_path: str) -> str:
    """
    Run Tesseract on a single line image.
    We use the Spanish+Catalan language pack and 
    page segmentation mode 7 (single line).
    """
    img = Image.open(image_path).convert('RGB')
    
    # PSM 7 = treat image as single text line
    # lang = spa (Spanish) — closest available to historical Catalan
    config = '--psm 7 -l spa'
    
    text = pytesseract.image_to_string(img, config=config)
    return text.strip()

In [ ]:
results = []

for record in train_records[:50]:
    parsed = parse_transcription(record.transcription)
    if not parsed:
        continue
    
    for line in record.lines:
        line_key = f'Line{line.line_idx}'
        reference = parsed.get(line_key)
        if not reference:
            continue
        
        # Run Tesseract
        hypothesis = ocr_line_tesseract(line.image_path)
        cer = compute_cer(hypothesis, reference)
        
        results.append({
            'record':     record.record_id,
            'line':       line_key,
            'reference':  reference,
            'hypothesis': hypothesis,
            'cer':        cer
        })

# Summary
avg_cer = sum(r['cer'] for r in results) / len(results)
print(f"Records evaluated : {len(set(r['record'] for r in results))}")
print(f"Lines evaluated   : {len(results)}")
print(f"Average CER       : {avg_cer:.3f}  ({avg_cer*100:.1f}%)")

print("\n--- 5 sample predictions ---")
for r in results[:5]:
    print(f"REF : {r['reference']}")
    print(f"HYP : {r['hypothesis']}")
    print(f"CER : {r['cer']:.3f}")
    print("---")

Records evaluated : 50
Lines evaluated   : 164
Average CER       : 0.687  (68.7%)

--- 5 sample predictions ---
REF : Pro al pr de dits mes y any rebere de las sposallas
HYP : <Fc¡ e |a !fwg_;e d el
CER : 0.824
---
REF : de Antoni Colomer assaunador de Bara fill de Genis
HYP : se Snumi ÚRoner framador Se Day Jat 5e Cora
CER : 0.560
---
REF : Colomer pages de Vilassar y de Antiga ab Hieronyma
HYP : 
CER : 1.000
---
REF : donsella filla de Onofre Esquer morraler de Bara y
HYP : Smplts la %e Orafio Gquer ml 50 Ta 9
CER : 0.580
---
REF : de Eularia
HYP : 9€v€7w(m;£w
CER : 1.100
---


In [19]:
# Save baseline results for later comparison
baseline_results = {
    'model': 'Tesseract (cat)',
    'records_evaluated': len(set(r['record'] for r in results)),
    'lines_evaluated': len(results),
    'avg_cer': avg_cer,
    'per_line': results
}

print("Baseline locked:")
print(f"  Model : {baseline_results['model']}")
print(f"  CER   : {baseline_results['avg_cer']:.3f}")

NameError: name 'results' is not defined

In [20]:
!pip install kraken --pre
print("Kraken installed")

Kraken installed



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import subprocess
import os

MODEL_PATH = (
    r"C:\Users\Abd Elrahman\AppData\Local\htrmopo\htrmopo"
    r"\78d7d26e-50b3-55f8-b839-c7a882ea2053"
    r"\McCATMuS_nfd_nofix_V1.mlmodel"
)

def ocr_line_kraken(image_path: str) -> str:
    """
    Run Kraken OCR on a single pre-cropped line image.
    --no-segmentation tells Kraken the whole image is one line.
    """
    output_path = 'kraken_temp_output.txt'
    
    cmd = [
        r'kraken_env\Scripts\kraken',  # path to kraken in venv
        '-i', image_path,
        output_path,
        'ocr',
        '--no-segmentation',
        '-m', MODEL_PATH
    ]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    if not os.path.exists(output_path):
        return ''
    
    with open(output_path, 'r', encoding='utf-8') as f:
        text = f.read().strip()
    
    os.remove(output_path)
    return text


# Test on first line
test_path = train_records[0].lines[0].image_path
result = ocr_line_kraken(test_path)
print(f"Kraken output : {result}")
print(f"Reference     : Pro al pr de dits mes y any rebere de las sposallas")

Kraken output : 
Reference     : Pro al pr de dits mes y any rebere de las sposallas


In [ ]:
kraken_results = []

for record in train_records[:50]:
    parsed = parse_transcription(record.transcription)
    if not parsed:
        continue
    
    for line in record.lines:
        line_key  = f'Line{line.line_idx}'
        reference = parsed.get(line_key)
        if not reference:
            continue
        
        hypothesis = ocr_line_kraken(line.image_path)
        cer        = compute_cer(hypothesis, reference)
        
        kraken_results.append({
            'record':     record.record_id,
            'line':       line_key,
            'reference':  reference,
            'hypothesis': hypothesis,
            'cer':        cer
        })

avg_cer_kraken = sum(r['cer'] for r in kraken_results) / len(kraken_results)

print(f"Lines evaluated    : {len(kraken_results)}")
print(f"Kraken CER         : {avg_cer_kraken:.3f} ({avg_cer_kraken*100:.1f}%)")
print(f"Tesseract CER      : 0.687 (68.7%)")
print(f"Difference         : {(avg_cer_kraken - 0.687):.3f}")

print("\n--- 5 sample predictions ---")
for r in kraken_results[:5]:
    print(f"REF    : {r['reference']}")
    print(f"KRAKEN : {r['hypothesis']}")
    print(f"CER    : {r['cer']:.3f}")
    print("---")

Lines evaluated    : 164
Kraken CER         : 0.542 (54.2%)
Tesseract CER      : 0.687 (68.7%)
Difference         : -0.145

--- 5 sample predictions ---
REF    : Pro al pr de dits mes y any rebere de las sposallas
KRAKEN : Duis d uirua
CER    : 0.863
---
REF    : de Antoni Colomer assaunador de Bara fill de Genis
KRAKEN : Ee Muri Ctorer essaraedn de Be^ Q^li de Goils,
CER    : 0.500
---
REF    : Colomer pages de Vilassar y de Antiga ab Hieronyma
KRAKEN : Cotour pres deMilgsus 9 ^e hgp, àt Fheaggine
CER    : 0.580
---
REF    : donsella filla de Onofre Esquer morraler de Bara y
KRAKEN : npets Alle be Orote Epus nemiles e Fbr^ )
CER    : 0.560
---
REF    : de Eularia
KRAKEN : Be Enlaria
CER    : 0.200
---


In [ ]:
baselines = {
    'tesseract': {
        'model':           'Tesseract (cat)',
        'cer':             0.687,
        'lines_evaluated': 164,
    },
    'kraken': {
        'model':           'McCATMuS (Kraken 7.0)',
        'cer':             avg_cer_kraken,
        'lines_evaluated': len(kraken_results),
        'per_line':        kraken_results
    }
}

print("=" * 40)
print("BASELINE SUMMARY")
print("=" * 40)
for name, b in baselines.items():
    print(f"{name:12}: {b['cer']:.3f} CER ({b['cer']*100:.1f}%)")
print("=" * 40)
print(f"Target    : beat {min(b['cer'] for b in baselines.values()):.3f} with VLM pipeline")

BASELINE SUMMARY
tesseract   : 0.687 CER (68.7%)
kraken      : 0.542 CER (54.2%)
Target    : beat 0.542 with VLM pipeline


In [22]:
import torch

print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
print(f"GPU name            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA version        : {torch.version.cuda}")

PyTorch version     : 2.5.1+cu121
CUDA available      : True
GPU name            : NVIDIA GeForce GTX 1650
CUDA version        : 12.1


In [23]:
import os
import shutil

def prepare_kraken_ground_truth(train_records: list,
                                 output_dir: str) -> dict:
    os.makedirs(output_dir, exist_ok=True)
    
    stats = {'matched': 0, 'unmatched': 0, 'skipped': 0}
    
    for record in train_records:
        if not record.transcription:
            stats['skipped'] += len(record.lines)
            continue
        
        parsed = parse_transcription(record.transcription)
        if not parsed:
            stats['skipped'] += len(record.lines)
            continue
        
        for line in record.lines:
            line_key = f'Line{line.line_idx}'
            ref_text = parsed.get(line_key)
            
            if not ref_text:
                stats['unmatched'] += 1
                continue
            
            img_filename = os.path.basename(line.image_path)
            base_name    = os.path.splitext(img_filename)[0]
            gt_path      = os.path.join(output_dir, f'{base_name}.gt.txt')
            img_dest     = os.path.join(output_dir, img_filename)
            
            # Write ground truth
            with open(gt_path, 'w', encoding='utf-8') as f:
                f.write(ref_text)
            
            # Copy image
            shutil.copy2(line.image_path, img_dest)
            stats['matched'] += 1
    
    return stats


output_dir = r'D:\GSoc 2026\Human AI - RenAIssance Test1\Data\kraken_training'

print("Preparing ground truth files...")
stats = prepare_kraken_ground_truth(train_records, output_dir)

print(f"Matched   : {stats['matched']}")
print(f"Unmatched : {stats['unmatched']}")
print(f"Skipped   : {stats['skipped']}")

# Verify
files = os.listdir(output_dir)
png_count = sum(1 for f in files if f.endswith('.png'))
gt_count  = sum(1 for f in files if f.endswith('.gt.txt'))
print(f"\nFiles in output dir:")
print(f"  PNG files    : {png_count}")
print(f"  GT txt files : {gt_count}")
print(f"  Pairs match  : {png_count == gt_count}")

Preparing ground truth files...
Matched   : 3070
Unmatched : 0
Skipped   : 0

Files in output dir:
  PNG files    : 3070
  GT txt files : 3070
  Pairs match  : True
